In [1]:
# IMPORTS
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)

In [2]:
# MODEL: P01
# ONLY CHANGE: a0, b0 hierarchical (Half-Cauchy + MH)
# EVERYTHING ELSE KEPT AS YOUR WEEKLY VERSION, INCLUDING tqdm + display

# ================================================================
# Imports
# ================================================================
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix, diags, bmat
from scipy.sparse.csgraph import connected_components
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import geopandas as gpd
import pyreadr
import pickle
from IPython.display import clear_output, display

# ================================================================
# Load data (remove isolated points)
# ================================================================
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0]

all_y = snow.drop(index=no_nbs).reset_index(drop=True)
coords_full = all_y.iloc[:, :2].to_numpy()
y_full      = all_y.iloc[:, 2:].to_numpy()

S_full, TT = y_full.shape
period = 52

# ================================================================
# Global time trend (scaled once)
# ================================================================
t_full = np.arange(1, TT + 1)
t_trend_full = (t_full - t_full.mean()) / t_full.std(ddof=0)

# ================================================================
# Build adjacency on FULL graph
# ================================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_full[:, 0], coords_full[:, 1]),
    crs="EPSG:4326"
)
gdf = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy_full = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6

A_full = (squareform(pdist(xy_full)) <= 0.22).astype(int)
np.fill_diagonal(A_full, 0)
A_full = csr_matrix(A_full)

# ================================================================
# Merge two largest connected components
# ================================================================
n_comp, labels = connected_components(A_full, directed=False)
sizes = np.bincount(labels)
comp1, comp2 = np.argsort(sizes)[-2:][::-1]

keep = np.where((labels == comp1) | (labels == comp2))[0]
# keep = np.where((labels == comp1))[0]
coords = coords_full[keep]
xy     = xy_full[keep]
y      = y_full[keep]
S      = y.shape[0]

# ================================================================
# Rebuild adjacency (merged)
# ================================================================
A = (squareform(pdist(xy)) <= 0.22).astype(int)
np.fill_diagonal(A, 0)
A = csr_matrix(A)

deg = np.array(A.sum(axis=1)).flatten()
Q_icar = diags(deg) - A
I_S    = diags(np.ones(S))

# ================================================================
# WEEKLY bins (1..52)
# ================================================================
time_bin = 52
# bin_edges = np.array([0, 13, 26, 39, 52])
bin_edges = np.arange(53)
# bin_edges = [1,52]
bin_str = "weekly_1_52"

# ================================================================
# p01 mask
# ================================================================
loc_mask = (y[:, :-1] == 0)
row_idx, time_idx = np.where(loc_mask)
N0 = len(row_idx)

outcome = y[row_idx, time_idx + 1]
kappa = outcome - 0.5

t_raw   = time_idx + 1
t_trend = t_trend_full[time_idx]

week_in_year = ((t_raw - 1) % 52) + 1

# ================================================================
# 4 base covariates (intercept, cos, sin, trend)
# ================================================================
cov4 = np.column_stack([
    np.ones(N0),
    np.cos(2*np.pi*week_in_year/period),
    np.sin(2*np.pi*week_in_year/period),
    t_trend
])

K_base = cov4.shape[1]   # 4
R = 2                    # ICAR + IID
K_total = K_base * R     # 8

# ================================================================
# Build X_eta_base   (N0 × 8S)
# ================================================================

rows_e, cols_e, vals_e = [], [], []

for i in tqdm(range(N0), desc="Building X_eta_base", leave=False):

    s = row_idx[i]

    for k in range(K_base):
        xval = cov4[i, k]

        # ICAR component
        col_icar = (2*k) * S + s
        rows_e.append(i)
        cols_e.append(col_icar)
        vals_e.append(xval)

        # IID component
        col_iid = (2*k+1) * S + s
        rows_e.append(i)
        cols_e.append(col_iid)
        vals_e.append(xval)

X_eta_base = coo_matrix(
    (vals_e, (rows_e, cols_e)),
    shape=(N0, K_total * S)
).tocsr()

# ================================================================
# Build X_tau_base   (N0 × 8T)
# ================================================================
week_idx = (t_raw - 1) % 52   # 0..51
rows_t, cols_t, vals_t = [], [], []
T_week = 52

for i in tqdm(range(N0), desc="Building X_tau_base", leave=False):

    w = week_idx[i]   # 0..51

    for k in range(K_base):
        xval = cov4[i, k]

        # ICAR loading
        col_icar = (2*k) * T_week + w
        rows_t.append(i)
        cols_t.append(col_icar)
        vals_t.append(xval)

        # IID loading
        col_iid = (2*k+1) * T_week + w
        rows_t.append(i)
        cols_t.append(col_iid)
        vals_t.append(xval)

X_tau_base = coo_matrix(
    (vals_t, (rows_t, cols_t)),
    shape=(N0, K_total * T_week)
).tocsr()

In [5]:
eta_dim = K_total * S
tau_dim = K_total * T_week

Q_blocks = []

for k in range(K_base):
    # ICAR
    Q_blocks.append(Q_icar)
    # IID
    Q_blocks.append(I_S)

Q_eta = bmat(
    [[Q_blocks[i] if i == j else None
      for j in range(K_total)]
     for i in range(K_total)],
    format="csr"
)


tau_prior_prec = diags(np.ones(tau_dim))/3

# Initialization

curr_eta = np.ones(eta_dim)*10
curr_tau = np.ones(tau_dim)*10

burn, thin, tot_save = 1000, 5, 1000
total_iters = burn + thin * tot_save
all_eta = np.zeros((eta_dim, tot_save))
all_tau = np.zeros((tau_dim, tot_save))

save_idx = 0


In [6]:
for it in tqdm(range(total_iters), desc="MCMC | p01 factor"):

    # ============================================================
    # 1️⃣ Build full linear predictor psi (using current eta & tau)
    # ============================================================

    # Build X_tilde for eta block (tau absorbed)
    X_tilde_eta = X_eta_base.copy()
    rows, cols = X_tilde_eta.nonzero()

    j = cols // S
    w = week_idx[rows]
    tau_indices = j * 52 + w

    X_tilde_eta.data *= curr_tau[tau_indices]

    psi = X_tilde_eta @ curr_eta   # FULL psi


    # ============================================================
    # 2️⃣ Sample omega ONCE
    # ============================================================

    omega = random_polyagamma(1, psi)


    # ============================================================
    # 3️⃣ Update eta | tau, omega
    # ============================================================

    XtOmega = X_tilde_eta.T.multiply(omega)
    post_prec_eta = XtOmega @ X_tilde_eta + Q_eta
    post_prec_eta = (post_prec_eta + post_prec_eta.T) * 0.5

    rhs_eta = X_tilde_eta.T @ kappa

    factor = cholesky(post_prec_eta, mode="simplicial")

    mu = factor.solve_A(rhs_eta)

    z = np.random.randn(eta_dim)
    z = z / np.sqrt(factor.D())
    z = factor.solve_Lt(z)
    z = factor.apply_Pt(z)

    curr_eta = mu + z


    # ============================================================
    # 4️⃣ Update tau | eta, SAME omega
    # ============================================================

    # Rebuild X_tilde_tau (eta absorbed)
    X_tilde_tau = X_tau_base.copy()
    rows, cols = X_tilde_tau.nonzero()

    j = cols // 52
    s = row_idx[rows]
    eta_indices = j * S + s

    X_tilde_tau.data *= curr_eta[eta_indices]

    XtOmega = X_tilde_tau.T.multiply(omega)
    post_prec_tau = XtOmega @ X_tilde_tau + tau_prior_prec
    post_prec_tau = (post_prec_tau + post_prec_tau.T) * 0.5

    rhs_tau = X_tilde_tau.T @ kappa

    factor = cholesky(post_prec_tau, mode="simplicial")

    mu = factor.solve_A(rhs_tau)

    z = np.random.randn(tau_dim)
    z = z / np.sqrt(factor.D())
    z = factor.solve_Lt(z)
    z = factor.apply_Pt(z)

    curr_tau = mu + z


    # ============================================================
    # (Optional) Monitor scale
    # ============================================================

    # print(np.max(np.abs(curr_tau)), np.max(np.abs(curr_eta)))

    # ============================================================
    # Display tau (monitor convergence)
    # ============================================================

    # if it % 5 == 0:

    #     tau_mat = curr_tau.reshape(K_total, 52).T  # 52 × 8

    #     df_tau = pd.DataFrame(
    #         tau_mat,
    #         index=[f"w{w+1}" for w in range(52)],
    #         columns=[f"comp_{j}" for j in range(K_total)]
    #     )

    #     clear_output(wait=True)
    #     print(f"Iteration {it}")
    #     print("max |tau|:", np.max(np.abs(curr_tau)))
    #     print("min tau:", np.min(curr_tau))
    #     print("max tau:", np.max(curr_tau))
    #     display(df_tau)
    # ============================================================
    # Save
    # ============================================================
    # print(save_idx)
    if it >= burn and (it - burn) % thin == 0:

        all_eta[:, save_idx] = curr_eta
        all_tau[:, save_idx] = curr_tau

        save_idx += 1
        if save_idx == tot_save:
            break
# ============================================================
# SAVE p01 results
# ============================================================

save_dict = {
    "all_eta": all_eta,
    "all_tau": all_tau,
    "S": S,
    "TT": TT,
    "keep_index": keep,
    "bin_str": bin_str,
    "transition": "p01"
}

filename = f"BYM_factor_twoComp_p01.pkl"

with open(filename, "wb") as f:
    pickle.dump(save_dict, f)

print(f"\nSaved p01 results to {filename}")

MCMC | p01 factor:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_6812\3115132114.py:37: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(post_prec_eta, mode="simplicial")
C:\Users\lix23\AppData\Local\Temp\ipykernel_6812\3115132114.py:69: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(post_prec_tau, mode="simplicial")
MCMC | p01 factor: 100%|█████████▉| 5995/6000 [8:02:13<00:24,  4.83s/it]   


Saved p01 results to BYM_factor_twoComp_p01.pkl


In [1]:
# ================================================================
# Imports
# ================================================================
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import coo_matrix, csr_matrix, bmat, diags
from scipy.sparse.csgraph import connected_components
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import pyreadr
import pickle

# ================================================================
# Load data
# ================================================================
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords = snow.iloc[:, :2].to_numpy()
y = snow.iloc[:, 2:].to_numpy()

# ================================================================
# Keep TWO largest connected components
# ================================================================
DIST_TH = 0.22

gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:,0], coords[:,1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6
Dmat = squareform(pdist(xy))

W = (Dmat <= DIST_TH).astype(int)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

n_comp, labels = connected_components(W, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

use_idx = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

coords = coords[use_idx]
y = y[use_idx]
W = W[use_idx][:, use_idx]

S, TT = y.shape
period = 52

print("Using S =", S)

# ================================================================
# Build CAR precision
# ================================================================
D = csr_matrix(np.diag(np.array(W.sum(axis=1)).flatten()))
Q_car = D - W

# ================================================================
# Build p10 dataset (1 → 0)
# ================================================================
loc = np.where(y[:, :-1] == 1)
pairs = np.column_stack(loc)
pairs = pairs[np.lexsort((pairs[:,0], pairs[:,1]))]
pairs[:,1] += 1

row_idx = pairs[:,0]
time_idx = pairs[:,1] - 1
N = len(row_idx)

next_y = y[pairs[:,0], pairs[:,1]]
kappa = (1 - next_y) - 0.5

# ================================================================
# Covariates (8 components)
# ================================================================
t_raw = time_idx + 1
t_scaled = (t_raw - t_raw.mean()) / t_raw.std()

covariates = np.column_stack([
    np.ones(N), np.ones(N),
    np.cos(2*np.pi*t_raw/period), np.cos(2*np.pi*t_raw/period),
    np.sin(2*np.pi*t_raw/period), np.sin(2*np.pi*t_raw/period),
    t_scaled, t_scaled
])

K_total = covariates.shape[1]

# ================================================================
# Build X_eta_base
# ================================================================
rows, cols, vals = [], [], []

for i in range(N):
    s = row_idx[i]
    for k in range(K_total):
        rows.append(i)
        cols.append(s + k*S)
        vals.append(covariates[i,k])

X_eta_base = coo_matrix(
    (vals,(rows,cols)),
    shape=(N, K_total*S)
).tocsr()

eta_dim = K_total * S

# ================================================================
# Build X_tau_base
# tau dimension = K_total × 52
# ================================================================
week_idx = time_idx % 52

tau_dim = K_total * 52

rows, cols, vals = [], [], []

for i in range(N):
    for k in range(K_total):
        rows.append(i)
        cols.append(k*52 + week_idx[i])
        vals.append(covariates[i,k])

X_tau_base = coo_matrix(
    (vals,(rows,cols)),
    shape=(N, tau_dim)
).tocsr()

# ================================================================
# Priors
# ================================================================
# eta prior precision (BYM structure)

blocks = []
for k in range(K_total):
    if k % 2 == 0:
        blocks.append(Q_car)
    else:
        blocks.append(diags(np.ones(S)))

Q_eta = bmat([[blocks[i] if i==j else None
               for j in range(K_total)]
               for i in range(K_total)],
              format="csr")

tau_prior_prec = diags(np.ones(tau_dim)/3)

# ================================================================
# MCMC settings
# ================================================================
burn = 1000
thin = 5
tot_save = 1000
total_iters = burn + tot_save * thin

all_eta = np.zeros((eta_dim, tot_save))
all_tau = np.zeros((tau_dim, tot_save))

curr_eta = np.zeros(eta_dim)
curr_tau = np.ones(tau_dim)

save_idx = 0

# ================================================================
# MCMC
# ================================================================
for it in tqdm(range(total_iters), desc="MCMC | p10 factor"):

    # ---- build psi
    X_tilde_eta = X_eta_base.copy()
    rows, cols = X_tilde_eta.nonzero()

    j = cols // S
    w = week_idx[rows]
    tau_indices = j*52 + w

    X_tilde_eta.data *= curr_tau[tau_indices]
    psi = X_tilde_eta @ curr_eta

    # ---- PG
    omega = random_polyagamma(1, psi)

    # ---- eta update
    XtOmega = X_tilde_eta.T.multiply(omega)
    post_prec_eta = XtOmega @ X_tilde_eta + Q_eta
    post_prec_eta = (post_prec_eta + post_prec_eta.T)*0.5
    post_prec_eta = post_prec_eta.tocsc()

    rhs_eta = X_tilde_eta.T @ kappa

    factor = cholesky(post_prec_eta, mode="simplicial")
    mu = factor.solve_A(rhs_eta)

    z = np.random.randn(eta_dim)
    z = z / np.sqrt(factor.D())
    z = factor.solve_Lt(z)
    z = factor.apply_Pt(z)

    curr_eta = mu + z

    # ---- tau update
    X_tilde_tau = X_tau_base.copy()
    rows, cols = X_tilde_tau.nonzero()

    j = cols // 52
    s = row_idx[rows]
    eta_indices = j*S + s

    X_tilde_tau.data *= curr_eta[eta_indices]

    XtOmega = X_tilde_tau.T.multiply(omega)
    post_prec_tau = XtOmega @ X_tilde_tau + tau_prior_prec
    post_prec_tau = (post_prec_tau + post_prec_tau.T)*0.5
    post_prec_tau = post_prec_tau.tocsc()

    rhs_tau = X_tilde_tau.T @ kappa

    factor = cholesky(post_prec_tau, mode="simplicial")
    mu = factor.solve_A(rhs_tau)

    z = np.random.randn(tau_dim)
    z = z / np.sqrt(factor.D())
    z = factor.solve_Lt(z)
    z = factor.apply_Pt(z)

    curr_tau = mu + z

    # ---- save
    if it >= burn and (it - burn) % thin == 0:
        all_eta[:, save_idx] = curr_eta
        all_tau[:, save_idx] = curr_tau
        save_idx += 1
        if save_idx == tot_save:
            break

# ================================================================
# Save
# ================================================================
save_dict = {
    "all_eta": all_eta,
    "all_tau": all_tau,
    "S": S,
    "TT": TT,
    "transition": "p10"
}

with open("BYM_factor_twoComp_p10.pkl", "wb") as f:
    pickle.dump(save_dict, f)

print("Saved BYM_factor_twoComp_p10.pkl")

Using S = 1557


MCMC | p10 factor: 100%|█████████▉| 5995/6000 [4:17:08<00:12,  2.57s/it]  

Saved BYM_factor_twoComp_p10.pkl


In [ ]:
# ================================================================
# Factor BYM (scale-varying) posterior mean LLH
# From scratch: load y -> keep 2 largest components -> build X_eta_base
# -> load posterior (eta,tau) from pkl -> compute E_post[ loglik ]
#
# NOTE:
# - This matches your TRAINING design: 8S columns via duplicated covariates:
#   [1,1, cos,cos, sin,sin, t,t]
# - Likelihood uses y_vec directly (NO kappa).
# - psi is built exactly like training: absorb tau into X data, then X@eta.
# ================================================================

import numpy as np
import pickle
import geopandas as gpd
import pyreadr
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix
from scipy.sparse.csgraph import connected_components

# ================================================================
# 0) Config
# ================================================================
DIST_TH = 0.22
period = 52

no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

# ================================================================
# 1) Load data -> drop isolated points
# ================================================================
snow = pyreadr.read_r("snow_cleaned_full.Rda")
snow = list(snow.values())[0].reset_index(drop=True)
snow = snow.drop(index=no_nbs).reset_index(drop=True)

coords_all = snow.iloc[:, :2].to_numpy()
y_all = snow.iloc[:, 2:].to_numpy()

S0, TT = y_all.shape
print("Loaded full (after no_nbs): S0 =", S0, "TT =", TT)

# ================================================================
# 2) Build adjacency W on ALL remaining points (for components)
#    (same AEQD projection + distance threshold)
# ================================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords_all[:, 0], coords_all[:, 1]),
    crs="EPSG:4326"
).to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")

xy = np.vstack([gdf.geometry.x, gdf.geometry.y]).T / 1e6  # million meters, consistent with your code
Dmat = squareform(pdist(xy))

W = (Dmat <= DIST_TH).astype(np.int8)
np.fill_diagonal(W, 0)
W = csr_matrix(W)

n_comp, labels = connected_components(W, directed=False)
sizes = np.bincount(labels)
order = np.argsort(sizes)[::-1]

print("n_comp =", n_comp)
print("Top 10 component sizes:", sizes[order[:10]])

# ================================================================
# 3) Keep TWO largest connected components
# ================================================================
use_idx = np.sort(np.concatenate([
    np.where(labels == order[0])[0],
    np.where(labels == order[1])[0]
]))

coords = coords_all[use_idx]
y = y_all[use_idx]
W2 = W[use_idx][:, use_idx]

S, TT2 = y.shape
print("Keeping 2 largest components: S =", S, "TT =", TT2)

# ================================================================
# 4) Global scaled trend (same definition as your BYM code)
# ================================================================
t_full = np.arange(1, TT2 + 1)
t_scaled = (t_full - t_full.mean()) / t_full.std(ddof=0)

# ================================================================
# 5) ---------- p01: build design X_eta_base01 (N01 x 8S), y_vec01 ----------
#     p01 event: y_t=0 -> y_{t+1}=1
# ================================================================
loc = np.where(y[:, :-1] == 0)
pairs = np.column_stack(loc)
pairs = pairs[np.lexsort((pairs[:, 0], pairs[:, 1]))]
pairs[:, 1] += 1

row_idx01  = pairs[:, 0]
time_idx01 = pairs[:, 1] - 1
N01 = len(row_idx01)

next_y01 = y[pairs[:, 0], pairs[:, 1]]
y_vec01 = next_y01.astype(np.float64)  # success prob is for next_y=1

week_idx01 = (time_idx01 % period).astype(np.int64)
t01 = t_scaled[time_idx01]

# duplicated covariates: [1,1, cos,cos, sin,sin, t,t]
cov01 = np.column_stack([
    np.ones(N01), np.ones(N01),
    np.cos(2*np.pi*(time_idx01 + 1)/period),
    np.cos(2*np.pi*(time_idx01 + 1)/period),
    np.sin(2*np.pi*(time_idx01 + 1)/period),
    np.sin(2*np.pi*(time_idx01 + 1)/period),
    t01, t01
])

K_total = cov01.shape[1]  # MUST be 8
eta_dim = K_total * S
print("\n[p01] N01 =", N01, "| K_total =", K_total, "| eta_dim =", eta_dim)

rows, cols, vals = [], [], []
for i in range(N01):
    s = row_idx01[i]
    for j in range(K_total):
        rows.append(i)
        cols.append(j * S + s)
        vals.append(cov01[i, j])

X_eta_base01 = coo_matrix((vals, (rows, cols)), shape=(N01, eta_dim)).tocsr()

# ================================================================
# 6) ---------- p10: build design X_eta_base10 (N10 x 8S), y_vec10 ----------
#     p10 event: y_t=1 -> y_{t+1}=0  (success = 1-next_y)
# ================================================================
loc = np.where(y[:, :-1] == 1)
pairs = np.column_stack(loc)
pairs = pairs[np.lexsort((pairs[:, 0], pairs[:, 1]))]
pairs[:, 1] += 1

row_idx10  = pairs[:, 0]
time_idx10 = pairs[:, 1] - 1
N10 = len(row_idx10)

next_y10 = y[pairs[:, 0], pairs[:, 1]]
y_vec10 = (1 - next_y10).astype(np.float64)  # success prob is for next_y=0

week_idx10 = (time_idx10 % period).astype(np.int64)
t10 = t_scaled[time_idx10]

cov10 = np.column_stack([
    np.ones(N10), np.ones(N10),
    np.cos(2*np.pi*(time_idx10 + 1)/period),
    np.cos(2*np.pi*(time_idx10 + 1)/period),
    np.sin(2*np.pi*(time_idx10 + 1)/period),
    np.sin(2*np.pi*(time_idx10 + 1)/period),
    t10, t10
])

print("\n[p10] N10 =", N10, "| K_total =", cov10.shape[1], "| eta_dim =", eta_dim)

rows, cols, vals = [], [], []
for i in range(N10):
    s = row_idx10[i]
    for j in range(K_total):
        rows.append(i)
        cols.append(j * S + s)
        vals.append(cov10[i, j])

X_eta_base10 = coo_matrix((vals, (rows, cols)), shape=(N10, eta_dim)).tocsr()

# ================================================================
# 7) Load posterior samples (factor model)
# ================================================================
with open(base"BYM_factor_twoComp_p01.pkl", "rb") as f:
    res01 = pickle.load(f)
with open("BYM_factor_twoComp_p10.pkl", "rb") as f:
    res10 = pickle.load(f)

all_eta01 = res01["all_eta"]
all_tau01 = res01["all_tau"]
all_eta10 = res10["all_eta"]
all_tau10 = res10["all_tau"]

M01 = all_eta01.shape[1]
M10 = all_eta10.shape[1]
print("\nLoaded posterior draws:")
print("p01 all_eta:", all_eta01.shape, "all_tau:", all_tau01.shape)
print("p10 all_eta:", all_eta10.shape, "all_tau:", all_tau10.shape)

# sanity checks
assert all_eta01.shape[0] == eta_dim, f"p01 eta_dim mismatch: {all_eta01.shape[0]} vs {eta_dim}"
assert all_eta10.shape[0] == eta_dim, f"p10 eta_dim mismatch: {all_eta10.shape[0]} vs {eta_dim}"
assert all_tau01.shape[0] == K_total * period, f"p01 tau_dim mismatch: {all_tau01.shape[0]} vs {K_total*period}"
assert all_tau10.shape[0] == K_total * period, f"p10 tau_dim mismatch: {all_tau10.shape[0]} vs {K_total*period}"

# ================================================================
# 8) Posterior mean LLH for p01
#    psi_m = (X_eta_base01 with tau absorbed) @ eta_m
#    llh_m = sum( y*psi - log(1+exp(psi)) )
#    Use stable softplus: log(1+exp(psi)) = log1p(exp(-|psi|)) + max(psi,0)
# ================================================================
base_rows01, base_cols01 = X_eta_base01.nonzero()
j_idx01 = (base_cols01 // S).astype(np.int64)

llh_draws01 = np.zeros(M01)

for m in tqdm(range(M01), desc="LLH p01 over posterior draws"):
    eta_m = all_eta01[:, m]
    tau_m = all_tau01[:, m]

    X_tilde = X_eta_base01.copy()

    w_idx = week_idx01[base_rows01]
    tau_indices = j_idx01 * period + w_idx
    X_tilde.data *= tau_m[tau_indices]

    psi = X_tilde @ eta_m

    softplus = np.log1p(np.exp(-np.abs(psi))) + np.maximum(psi, 0.0)
    llh_draws01[m] = np.sum(y_vec01 * psi - softplus)

llh01 = llh_draws01.mean()
print("\nPosterior mean LLH p01:", llh01)

# ================================================================
# 9) Posterior mean LLH for p10
# ================================================================
base_rows10, base_cols10 = X_eta_base10.nonzero()
j_idx10 = (base_cols10 // S).astype(np.int64)

llh_draws10 = np.zeros(M10)

for m in tqdm(range(M10), desc="LLH p10 over posterior draws"):
    eta_m = all_eta10[:, m]
    tau_m = all_tau10[:, m]

    X_tilde = X_eta_base10.copy()

    w_idx = week_idx10[base_rows10]
    tau_indices = j_idx10 * period + w_idx
    X_tilde.data *= tau_m[tau_indices]

    psi = X_tilde @ eta_m

    softplus = np.log1p(np.exp(-np.abs(psi))) + np.maximum(psi, 0.0)
    llh_draws10[m] = np.sum(y_vec10 * psi - softplus)

llh10 = llh_draws10.mean()
print("\nPosterior mean LLH p10:", llh10)

# ================================================================
# 10) Total
# ================================================================
print("\nTOTAL posterior mean LLH (p01+p10):", llh01 + llh10)

Loaded full (after no_nbs): S0 = 1601 TT = 2704
n_comp = 14
Top 10 component sizes: [1068  489   10    6    6    4    3    3    2    2]
Keeping 2 largest components: S = 1557 TT = 2704

[p01] N01 = 2716873 | K_total = 8 | eta_dim = 12456

[p10] N10 = 1491698 | K_total = 8 | eta_dim = 12456


FileNotFoundError: [Errno 2] No such file or directory: 'BYM_factor_twoComp_p01.pkl'